In [6]:
import glob,json
import pandas as pd
from croniter import croniter
from datetime import datetime
from cron_descriptor import get_description, ExpressionDescriptor
pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', None)


# Set the CSS style for the DataFrame display
pd.set_option('display.max_colwidth', 100)


## run_etl.json

In [43]:
def find_files_recursive(directory):
    # This function finds all files recursively in a given directory
    return glob.glob(f'{directory}/**/run_etl.json', recursive=True)

def decodeExtract(extract):
    
    prog=""
    lang=""
    if 'language' in extract:
        lang=extract['language']
            
    if 'file' in extract:
        prog=extract['file']
        
    else:
        print("No Prgram",lang,title,file)
    opts={}
    if 'options' in extract:
        lang=extract['language']
        for opt in extract['options']:
            opt=opt.strip()
            indx=opt.find(" ")
            o=opt[:indx]
            opts[o]= opt[indx+1:]

    return lang,prog,opts


def processRun_etls(): 
## Get all run_etl.json files
    directory_path = '/home/joe/bic_etl'
    files = find_files_recursive(directory_path)
    
    
    
    datasets={}
    noEx=0
    for file in files:
        ll=len(file)
        group=file[18:-13]
        fin=open(file)
        infos = json.load(fin)
        fin.close()
        title=""
        extract=""
        for info in infos: 
            if 'title' in info:
                title=info['title']
                datasets[title]={}
                datasets[title]['extract']={}
                datasets[title]['extract']['language']=''
                datasets[title]['extract']['program']=''
                datasets[title]['extract']['options']={}
                datasets[title]['group']=group
                datasets[title]['transform']={}
                datasets[title]['load']={}
                
                
                if 'extract' in info:
                    extract=info['extract']
                    if isinstance(extract,dict):
                       lang,prog,opts=decodeExtract(extract)
                       datasets[title]['extract']['language']=lang
                       datasets[title]['extract']['program']=prog
                       datasets[title]['extract']['options']=opts
                    elif isinstance(extract,list):
                        for infoL in extract: 
                            lang,prog,opts=decodeExtract(infoL)
                            datasets[title]['extract']['language']=lang
                            datasets[title]['extract']['program']=prog
                            datasets[title]['extract']['options']=opts
                    else:
                        print("UNK",type(info))
                if 'transform' in info:
                    
                    transform=info['transform']
                    if 'file' in transform:
                       datasets[title]['transform']['program']=transform['file']
                    else:
                       datasets[title]['transform']['program']=''
                else:
                    datasets[title]['transform']['program']=''

                if 'load' in info:
                    load=info['load']
                    if 'file' in load:
                       datasets[title]['load']['sij']=load['file']
                    else:
                       datasets[title]['load']['sij']=''

                    if 'type' in load:
                       datasets[title]['load']['type']=load['type']
                    else:
                       datasets[title]['load']['type']=''
                    if 'format' in load:
                       datasets[title]['load']['format']=load['format']
                    else:
                       datasets[title]['load']['format']=''
                        
            else:
                print("No Title",file)
    return datasets

datasets = processRun_etls()

No Title /home/joe/bic_etl/general/example/run_etl.json


In [44]:
datasets

{'CIM Catalog Download': {'extract': {'language': 'node',
   'program': 'general/scripts/request_url.js',
   'options': {'-u': 'https://data.colorado.gov/api/views/e7nm-tn2z/rows.csv?accessType=DOWNLOAD',
    '-f': 'catalog/data_source/inventory-$(date +%F).csv'}},
  'group': 'catalog',
  'transform': {'program': ''},
  'load': {}},
 'Housing Inventory: New Listing Count in Colorado': {'extract': {'language': '',
   'program': '',
   'options': {}},
  'group': 'fred',
  'transform': {'program': 'fred/scripts/fred_new_listing_extract_transform.py'},
  'load': {'sij': 'fred_housing_inventory.sij',
   'type': 'datasync',
   'format': 'csv'}},
 'Average Weekly & Hourly Earnings and Work Hours Across Major Industry Sectors in Colorado': {'extract': {'language': 'py',
   'program': 'bls/scripts/blsExtract.py',
   'options': {'-p': 'bls',
    '-f': 'sm.data.6.Colorado',
    '-o': 'data_source/sm.data.6.Colorado.tsv',
    '-u': 'https://download.bls.gov/pub/time.series/sm',
    '-r': 'Accept: 

### Get web/ftp Addresses

In [45]:
def getUrlAddresses(datasets):
    hist={}
    addresses={}
    options={}
    specOpts={}
    specOpts["general/scripts/request_url.js"]={}
    specOpts["general/scripts/request_url.js"]["-u"]="-f"
    
    specOpts["general/scripts/sftp_extract.js"]={}
    specOpts["general/scripts/sftp_extract.js"]["-h"]="-f"
    specOpts["general/scripts/sftp_extract.js"]["none"]="-f"
    
    specOpts["cdos/general/scripts/sftp_extract.js"]={}
    specOpts["cdos/general/scripts/sftp_extract.js"]["none"]="-f"
    
    lookupNotFound = {
    'bea/scripts/bea_msa_api_reader.py':['https://apps.bea.gov/api/data'],
    'denver/scripts/get_data.py':['https://www.denvergov.org/content/dam/denvergov/Portals/covid19/documents/Denver-Approved-Patio-Expansion-List.xlsx'],
    'scripts/extract_liquor_files.py':'https://sbg.colorado.gov/liquor-license-lists',
    'county_sales_extract.py':['https://docs.google.com/spreadsheets/d/1br_cwfHy24d2R2bcXacb2KarOIBKGrbR/export?format=csv'],
    'scripts/county_salestax_extract.py':['https://docs.google.com/spreadsheets/d/1EhlDIrXjJdmg_ob2eMFeN5xtyvTq2jtr/export?format=csv'],
    'scripts/tax_fee_extract.py':['https://docs.google.com/spreadsheets/d/1e8hnG64Vkg9ffgkNpKQ1kzPsXNbGjG2X/export?format=csv'],
    'scripts/state_mar_sales_extract.py':['https://docs.google.com/spreadsheets/d/1wE7Z_1q2zxL7-kP0AqlUvL731U3Aui1l/export?format=csv'],
    'scripts/city_extract.py':['https://docs.google.com/spreadsheets/d/1wo4WKzvlfdw47_3841jD2RUibbDwos3L/export?format=csv','https://docs.google.com/spreadsheets/d/1WI2jvRCzBqYFIPA2vsUmCjHedKCQD911/export?format=csv'],
    'scripts/county_extract.py':['https://docs.google.com/spreadsheets/d/1irLP8K7jREdDgIiamgZ2MiTbkeMX5lrM/export?format=csv'],
    'scripts/extract_scripts/citybyindustry_extract.py':['https://docs.google.com/spreadsheets/d/1ZKc0olDlChHyRiLlxL3ECxKqBYtyaUaB/export?format=csv','https://docs.google.com/spreadsheets/d/1E6VqKiPnUD3LMnrSBy1aEYSc4QsU6Y9v/export?format=csv'],
    'scripts/extract_scripts/industry_extract.py':['https://docs.google.com/spreadsheets/d/{1WANzZQFE57J73daXvo1xu4mITy-1DmuP/export?format=csv','https://docs.google.com/spreadsheets/d/1cmTJ4ZRAjBFfevT0hyCatUXDIzIG393I/export?format=csv'],
    'scripts/extract_scripts/countybyindustry_extract.py':['https://docs.google.com/spreadsheets/d/1CI66-qv0ooK93asc21VyV-tJiYSc2J3c/export?format=csv','https://docs.google.com/spreadsheets/d/1kybUGf02krqwyl8yBPHnTbn0iQT4Cnnz/export?format=csv'],
    'scripts/extract_liquor_files.py':['https://sbg.colorado.gov/liquor-license-lists']
    }
    
    
    noProg={}
    for title,dct in datasets.items():
        if 'group' in dct:
            group=dct['group']
        else:
            group=""
        if 'extract' in dct:
            if 'language' in dct['extract']:
                lang=dct['extract']['language']
                prog=dct['extract']['program']
                if lang not in hist:
                    hist[lang]={}
                if prog not in hist[lang]:
                    hist[lang][prog]=0
                hist[lang][prog]+=1
    
                if prog not in addresses:
                    addresses[prog]={}
                if 'options' in dct['extract']:
                    if prog in specOpts:
                        o=list(specOpts[prog].keys())[0]
                        fOpt=specOpts[prog][o]
                        ff = dct['extract']['options'][fOpt]  
                        if o in dct['extract']['options']:
                            a=dct['extract']['options'][o]
                        else:
                            if prog == "general/scripts/sftp_extract.js":
                              a="ftps.sos.state.co.us"
                            else:
                              a=""
                    else:
                        ff=""
                        o=""
                        a=""
                        if prog in lookupNotFound:
                            a = lookupNotFound[prog]
                        else:
                            if prog not in noProg:
                                noProg[prog]={}
                            noProg[prog]['group']=group
                        
                    if isinstance(a,str):
                        b=[]
                        b.append(a)
                    else:
                        b=a
                    for a in b: 
                        if a not in addresses[prog]:
                            addresses[prog][a]={}
                        if ff not in addresses[prog][a]:
                           addresses[prog][a][ff]={}
                        
                        if title not in addresses[prog][a][ff]:
                           addresses[prog][a][ff][title]={}
                            
                           addresses[prog][a][ff][title]['count']=0
                           addresses[prog][a][ff][title]['group']=group
                        
                        
                    addresses[prog][a][ff][title]['count']+=1
##  Create a DataFrame
    programs=[]
    address=[]
    files=[]
    counts=[]
    titles=[]
    groups=[]
    for p,dct in addresses.items():
        for o,dct2 in dct.items():   
            for f,dct3 in dct2.items():
                for t,dct4 in dct3.items():
                    
                    programs.append(p)
                    address.append(o)
                    files.append(f)
                    counts.append(dct4['count'])
                    groups.append(dct4['group'])
                    
                    titles.append(t)
    df = pd.DataFrame({
        "Title":titles,
        "Group":groups,
        "Program":programs,
        "Address":address,
        "File":files,
        "Count":counts})
    return addresses,df
addresses,df = getUrlAddresses(datasets)
        

In [46]:
datasets

{'CIM Catalog Download': {'extract': {'language': 'node',
   'program': 'general/scripts/request_url.js',
   'options': {'-u': 'https://data.colorado.gov/api/views/e7nm-tn2z/rows.csv?accessType=DOWNLOAD',
    '-f': 'catalog/data_source/inventory-$(date +%F).csv'}},
  'group': 'catalog',
  'transform': {'program': ''},
  'load': {}},
 'Housing Inventory: New Listing Count in Colorado': {'extract': {'language': '',
   'program': '',
   'options': {}},
  'group': 'fred',
  'transform': {'program': 'fred/scripts/fred_new_listing_extract_transform.py'},
  'load': {'sij': 'fred_housing_inventory.sij',
   'type': 'datasync',
   'format': 'csv'}},
 'Average Weekly & Hourly Earnings and Work Hours Across Major Industry Sectors in Colorado': {'extract': {'language': 'py',
   'program': 'bls/scripts/blsExtract.py',
   'options': {'-p': 'bls',
    '-f': 'sm.data.6.Colorado',
    '-o': 'data_source/sm.data.6.Colorado.tsv',
    '-u': 'https://download.bls.gov/pub/time.series/sm',
    '-r': 'Accept: 

In [47]:
def mapetload(row):
    title=row['Title']
    eprog=""
    elang=""
    tprog=""
    ltype=""
    lform=""
    sij=""
    if title in datasets:
        dct=datasets[title]
 
        if 'extract' in dct:
            if 'language' in dct['extract']:
                elang=dct['extract']['language']
                eprog= dct['extract']['program']
    
        if 'transform' in dct:
            if 'program' in dct['transform']:
                tprog=dct['transform']['program']

        if 'load' in dct:
            if 'sij' in dct['load']:
                sij=dct['load']['sij']
            if 'format' in dct['load']:
                lform=dct['load']['format']
            if 'type' in dct['load']:
                ltype=dct['load']['type']
                
        row['transform program']=tprog
        row['load type']=ltype
        row['load format']=lform
        row['load sij']=sij
    return row

df = df.apply(mapetload,axis=1)

In [ ]:
df.head(30)

## CRON

In [9]:
def getCronInfo(cron_file_path):
    fin=open(cron_file_path)
    lines=fin.readlines()
    crons={}
    for line in lines:
        if line[0:1] != "#" and line[0:1] != " " and len(line) > 2:
            line=line.strip()
            spl=line.split()
            crn=' '.join(spl[:5])
            rest = ' '.join(spl[5:])
            
            crons[rest]=crn
    return crons

def decodeCron(cron_file):
    hist={}
    cronGroups={}
    crons=getCronInfo(cron_file)
    for inf,crn in crons.items():
    
        desc = get_description(crn)
        print(desc)
        spl=inf.split()
        lang=spl[0]
        prg=spl[1]
        indxP=inf.find("-p")
        if indxP > -1:
           tmp=inf[indxP+2:].lstrip()
           end=tmp.find(" ")
           grp=tmp[:end]
        else:
            grp=""
    
        indxT=inf.find("-t")
        if indxT > -1:
           tmp=inf[indxT+2:].lstrip()
           start = tmp.find('"')
           end=tmp[start+1:].find('"')
           titl=tmp[start+1:end+1]
        else:
            titl="ALL"
        if grp not in cronGroups:
            cronGroups[grp]={}
        cronGroups[grp][titl]=desc
        print(grp,titl)     
        if lang not in hist:
            hist[lang]={}
        if prg not in hist[lang]:
            hist[lang][prg]=0
        hist[lang][prg]+=1
    return cronGroups
cron_file = '/home/joe/bic_etl/general/cron/cron_file'    
cronGroups=decodeCron(cron_file)


At 02:50 AM
 ALL
At 02:58 AM
 
At 05:00 AM
 ALL
At 04:00 AM, on day 15 of the month
bls/sm ALL
At 03:10 AM
boulder Septic Systems in Boulder County Colorado
At 04:00 AM
catalog ALL
At 06:05 AM, only on Friday
cdos/business/nonprofit ALL
At 05:05 AM
cdos/business/nonprofit Registration of Charities, Paid Solicitors, Professional Fundraising Consultants, and for-profit Public Benefit Corporations in Colorado
At 05:10 AM
cdos/business/business Business Entities in Colorado
At 08:00 AM
cdos/business/business Business Entity Transaction History
At 04:00 AM, only on Tuesday
cdos/health ALL
At 04:15 AM
cdos/lobbyist ALL
At 04:30 AM, only on Tuesday
cdos/government ALL
At 05:30 AM, on day 4 of the month
cdos/business/business Master List in Colorado
At 04:00 AM
cdos/business/business Trade Names for Businesses in Colorado
At 04:00 AM
cdos/business/business Trademarks for Businesses in Colorado
At 03:00 AM
cdos/business/ucc ALL
At 04:10 AM, on day 4 of the month
cdot/transportation_road_attribu

In [10]:
cronGroups

{'': {'ALL': 'At 02:47 PM', '': 'At 02:58 AM'},
 'bls/sm': {'ALL': 'At 04:00 AM, on day 15 of the month'},
 'boulder': {'Septic Systems in Boulder County Colorado': 'At 03:10 AM'},
 'catalog': {'ALL': 'At 04:00 AM'},
 'cdos/business/nonprofit': {'ALL': 'At 06:05 AM, only on Friday',
  'Registration of Charities, Paid Solicitors, Professional Fundraising Consultants, and for-profit Public Benefit Corporations in Colorado': 'At 05:05 AM'},
 'cdos/business/business': {'Business Entities in Colorado': 'At 05:10 AM',
  'Business Entity Transaction History': 'At 08:00 AM',
  'Master List in Colorado': 'At 05:30 AM, on day 4 of the month',
  'Trade Names for Businesses in Colorado': 'At 04:00 AM',
  'Trademarks for Businesses in Colorado': 'At 04:00 AM'},
 'cdos/health': {'ALL': 'At 04:00 AM, only on Tuesday'},
 'cdos/lobbyist': {'ALL': 'At 04:15 AM'},
 'cdos/government': {'ALL': 'At 04:30 AM, only on Tuesday'},
 'cdos/business/ucc': {'ALL': 'At 03:00 AM'},
 'cdot/transportation_road_attribut

## Combine Cron and Dataset Info

In [79]:
def addCronInfo(row):
    title=row['Title']
    group=row['Group']
    if group in cronGroups:
        dct=cronGroups[group]
        for how,crn in dct.items(): 
            if how == "ALL":
               cron=crn
            elif title in cronGroups[group]:
               cron=cronGroups[group][title]
            else:
                cron="Unknown"

    else:
        cron="No Group Found"
    return cron

df["Cron"] = df.apply(addCronInfo,axis=1)
    

In [ ]:
tmp=None

for prg,cnt in df['Program'].value_counts().to_dict().items():
    if tmp is not None:
       tmp=pd.concat([tmp,df.loc[df["Program"] == prg]])
    else:
       tmp=df.loc[df["Program"] == prg]
        
    display(prg)
    display(tmp[["Title","Group","Address"]].head(100))

In [19]:
df.columns

Index(['Title', 'Group', 'Program', 'Address', 'File', 'Count'], dtype='object')

In [52]:
tmp.to_csv("tilesInfo.csv",index=False)

In [18]:
df['Address'].value_counts()

Address
ftps.sos.state.co.us                                                                                                                                                                        46
                                                                                                                                                                                            17
prod.moveit.state.co.us                                                                                                                                                                      4
http://dtdapps.coloradodot.info/staticdata/Downloads/StateGeoData/Mileposts.zip                                                                                                              2
sftp.micropact.com                                                                                                                                                                           2
https://data.colorado.gov/api/views/e

In [81]:
df.head()

,Title,Group,Program,Address,File,Count,Cron
0,CIM Catalog Download,catalog,general/scripts/request_url.js,https://data.colorado.gov/api/views/e7nm-tn2z/rows.csv?accessType=DOWNLOAD,catalog/data_source/inventory-$(date +%F).csv,1,At 12:00 AM
1,Master List in Colorado,cdos/business/business,general/scripts/request_url.js,http://coloradosos.gov/pubs/UCC/downloadFiles/UCCMstrLB1.txt,cdos/business/business/data_source/masterlist.tsv,1,At 12:00 AM
2,Highway Milepoints in Colorado,cdot/transportation_road_attributes,general/scripts/request_url.js,http://dtdapps.coloradodot.info/staticdata/Downloads/StateGeoData/Mileposts.zip,cdot/transportation_road_attributes/data_source/mileposts.zip,1,At 12:00 AM
3,Highway Mileposts in Colorado,cdot/transportation_road_attributes,general/scripts/request_url.js,http://dtdapps.coloradodot.info/staticdata/Downloads/StateGeoData/Mileposts.zip,cdot/transportation_road_attributes/data_source/mileposts.zip,1,At 12:00 AM
4,Highway Routes in Colorado,cdot/transportation_road_attributes,general/scripts/request_url.js,http://dtdapps.coloradodot.info/staticdata/Downloads/StateGeoData/Routes.zip,cdot/transportation_road_attributes/data_source/routes.zip,1,At 12:00 AM


## All Combined Into ONE Cell

In [2]:
## Get ETL Directinves Out of run_etl.json For All Dataset

def find_files_recursive(directory):
    # This function finds all files recursively in a given directory
    return glob.glob(f'{directory}/**/run_etl.json', recursive=True)

def decodeExtract(extract):
    
    prog=""
    lang=""
    if 'language' in extract:
        lang=extract['language']
            
    if 'file' in extract:
        prog=extract['file']
        
    else:
        print("No Prgram",lang,title,file)
    opts={}
    if 'options' in extract:
        lang=extract['language']
        for opt in extract['options']:
            opt=opt.strip()
            indx=opt.find(" ")
            o=opt[:indx]
            opts[o]= opt[indx+1:]

    return lang,prog,opts


def processRun_etls(directory_path='/home/joe/bic_etl'): 
## Get all run_etl.json files
    files = find_files_recursive(directory_path)
    groups={}
    datasets={}
    noEx=0
    for file in files:
        ll=len(file)
        group=file[18:-13]
        fin=open(file)
        infos = json.load(fin)
        fin.close()
        title=""
        extract=""
        for info in infos: 
            if 'title' in info:
                title=info['title']
                datasets[title]={}
                datasets[title]['extract']={}
                datasets[title]['extract']['language']=''
                datasets[title]['extract']['program']=''
                datasets[title]['extract']['options']={}
                datasets[title]['group']=group
                if group not in groups:
                    groups[group]=[]
                groups[group].append(title)
                
                if 'extract' in info:
                    extract=info['extract']
                    if isinstance(extract,dict):
                       lang,prog,opts=decodeExtract(extract)
                       datasets[title]['extract']['language']=lang
                       datasets[title]['extract']['program']=prog
                       datasets[title]['extract']['options']=opts
                    elif isinstance(extract,list):
                        for infoL in extract: 
                            lang,prog,opts=decodeExtract(infoL)
                            datasets[title]['extract']['language']=lang
                            datasets[title]['extract']['program']=prog
                            datasets[title]['extract']['options']=opts
                    else:
                        print("UNK",type(info))
        
            else:
                print("No Title",file)
    return datasets,groups


def getUrlAddresses(datasets):
    hist={}
    addresses={}
    options={}
    specOpts={}
    specOpts["general/scripts/request_url.js"]={}
    specOpts["general/scripts/request_url.js"]["-u"]="-f"
    
    specOpts["general/scripts/sftp_extract.js"]={}
    specOpts["general/scripts/sftp_extract.js"]["-h"]="-f"
    specOpts["general/scripts/sftp_extract.js"]["none"]="-f"
    
    specOpts["cdos/general/scripts/sftp_extract.js"]={}
    specOpts["cdos/general/scripts/sftp_extract.js"]["none"]="-f"
    
    lookupNotFound = {
    'bea/scripts/bea_msa_api_reader.py':['https://apps.bea.gov/api/data'],
    'denver/scripts/get_data.py':['https://www.denvergov.org/content/dam/denvergov/Portals/covid19/documents/Denver-Approved-Patio-Expansion-List.xlsx'],
    'scripts/extract_liquor_files.py':'https://sbg.colorado.gov/liquor-license-lists',
    'county_sales_extract.py':['https://docs.google.com/spreadsheets/d/1br_cwfHy24d2R2bcXacb2KarOIBKGrbR/export?format=csv'],
    'scripts/county_salestax_extract.py':['https://docs.google.com/spreadsheets/d/1EhlDIrXjJdmg_ob2eMFeN5xtyvTq2jtr/export?format=csv'],
    'scripts/tax_fee_extract.py':['https://docs.google.com/spreadsheets/d/1e8hnG64Vkg9ffgkNpKQ1kzPsXNbGjG2X/export?format=csv'],
    'scripts/state_mar_sales_extract.py':['https://docs.google.com/spreadsheets/d/1wE7Z_1q2zxL7-kP0AqlUvL731U3Aui1l/export?format=csv'],
    'scripts/city_extract.py':['https://docs.google.com/spreadsheets/d/1wo4WKzvlfdw47_3841jD2RUibbDwos3L/export?format=csv','https://docs.google.com/spreadsheets/d/1WI2jvRCzBqYFIPA2vsUmCjHedKCQD911/export?format=csv'],
    'scripts/county_extract.py':['https://docs.google.com/spreadsheets/d/1irLP8K7jREdDgIiamgZ2MiTbkeMX5lrM/export?format=csv'],
    'scripts/extract_scripts/citybyindustry_extract.py':['https://docs.google.com/spreadsheets/d/1ZKc0olDlChHyRiLlxL3ECxKqBYtyaUaB/export?format=csv','https://docs.google.com/spreadsheets/d/1E6VqKiPnUD3LMnrSBy1aEYSc4QsU6Y9v/export?format=csv'],
    'scripts/extract_scripts/industry_extract.py':['https://docs.google.com/spreadsheets/d/{1WANzZQFE57J73daXvo1xu4mITy-1DmuP/export?format=csv','https://docs.google.com/spreadsheets/d/1cmTJ4ZRAjBFfevT0hyCatUXDIzIG393I/export?format=csv'],
    'scripts/extract_scripts/countybyindustry_extract.py':['https://docs.google.com/spreadsheets/d/1CI66-qv0ooK93asc21VyV-tJiYSc2J3c/export?format=csv','https://docs.google.com/spreadsheets/d/1kybUGf02krqwyl8yBPHnTbn0iQT4Cnnz/export?format=csv'],
    'scripts/extract_liquor_files.py':['https://sbg.colorado.gov/liquor-license-lists']
    }
    
    
    noProg={}
    for title,dct in datasets.items():
        if 'group' in dct:
            group=dct['group']
        else:
            group=""
        if 'extract' in dct:
            if 'language' in dct['extract']:
                lang=dct['extract']['language']
                prog=dct['extract']['program']
                if lang not in hist:
                    hist[lang]={}
                if prog not in hist[lang]:
                    hist[lang][prog]=0
                hist[lang][prog]+=1
    
                if prog not in addresses:
                    addresses[prog]={}
                if 'options' in dct['extract']:
                    if prog in specOpts:
                        o=list(specOpts[prog].keys())[0]
                        fOpt=specOpts[prog][o]
                        ff = dct['extract']['options'][fOpt]  
                        if o in dct['extract']['options']:
                            a=dct['extract']['options'][o]
                        else:
                            if prog == "general/scripts/sftp_extract.js":
                              a="ftps.sos.state.co.us"
                            else:
                              a=""
                    else:
                        ff=""
                        o=""
                        a=""
                        if prog in lookupNotFound:
                            a = lookupNotFound[prog]
                        else:
                            if prog not in noProg:
                                noProg[prog]={}
                            noProg[prog]['group']=group
                        
                    if isinstance(a,str):
                        b=[]
                        b.append(a)
                    else:
                        b=a
                    for a in b: 
                        if a not in addresses[prog]:
                            addresses[prog][a]={}
                        if ff not in addresses[prog][a]:
                           addresses[prog][a][ff]={}
                        
                        if title not in addresses[prog][a][ff]:
                           addresses[prog][a][ff][title]={}
                            
                           addresses[prog][a][ff][title]['count']=0
                           addresses[prog][a][ff][title]['group']=group
                        
                        
                    addresses[prog][a][ff][title]['count']+=1
##  Create a DataFrame
    programs=[]
    address=[]
    files=[]
    counts=[]
    titles=[]
    groups=[]
    for p,dct in addresses.items():
        for o,dct2 in dct.items():   
            for f,dct3 in dct2.items():
                for t,dct4 in dct3.items():
                    
                    programs.append(p)
                    address.append(o)
                    files.append(f)
                    counts.append(dct4['count'])
                    groups.append(dct4['group'])
                    
                    titles.append(t)
    df = pd.DataFrame({
        "Title":titles,
        "Group":groups,
        "Program":programs,
        "Address":address,
        "File":files,
        "Count":counts})
    return addresses,df

## Collect CRON Info from cron File

def getCronInfo(cron_file_path):
    fin=open(cron_file_path)
    lines=fin.readlines()
    crons={}
    for line in lines:
        if line[0:1] != "#" and line[0:1] != " " and len(line) > 2:
            line=line.strip()
            spl=line.split()
            crn=' '.join(spl[:5])
            rest = ' '.join(spl[5:])
            
            crons[rest]=crn
    return crons

def decodeCron(cron_file):
    hist={}
    cronGroups={}
    crons=getCronInfo(cron_file)
    for inf,crn in crons.items():
    
        desc = get_description(crn)
        spl=inf.split()
        lang=spl[0]
        prg=spl[1]
        indxP=inf.find("-p")
        if indxP > -1:
           tmp=inf[indxP+2:].lstrip()
           end=tmp.find(" ")
           grp=tmp[:end]
        else:
            grp=""
        indxT=inf.find("-t")
        if indxT > -1:
           tmp=inf[indxT+2:].lstrip()
           start = tmp.find('"')
           end=tmp[start+1:].find('"')
           titl=tmp[start+1:end+1]
        else:
            titl="ALL"
        if grp not in cronGroups:
            cronGroups[grp]={}
        cronGroups[grp][titl]=desc
        print(grp,titl)     
        if lang not in hist:
            hist[lang]={}
        if prg not in hist[lang]:
            hist[lang][prg]=0
        hist[lang][prg]+=1
    return cronGroups

## Combine Dataset ETL Info and CRON Info 
def addCronInfo(row):
    title=row['Title']
    group=row['Group']
#    print(group,title)
    if group in cronGroups:
        dct=cronGroups[group]
        if title in dct:
            cron=cronGroups[group][title]
        elif "ALL" in dct:
            cron=cronGroups[group]["ALL"]
        else:
            cron="Unknown"   
            
        # for how,crn in dct.items(): 
        #     print("  ",how,crn)
        #     if title in cronGroups[group]:
        #        cron=cronGroups[group][title]
        #        break
        #     elif how == "ALL":
        #         cron=crn
        #         break
        #     else:
        #         cron="Unknown"
        #         print("    Unknown ",how,title,crn)

    else:
        cron="No Group Found"
    print(group,cron,title)
    return cron

directory_path="/home/joe/bic_etl"
datasets,groups = processRun_etls(directory_path=directory_path)
addresses,df = getUrlAddresses(datasets) 
cron_file = '/home/joe/bic_etl/general/cron/cron_file'    
cronGroups=decodeCron(cron_file)
df["Cron"] = df.apply(addCronInfo,axis=1)

No Title /home/joe/bic_etl/general/example/run_etl.json
 ALL
 
 ALL
bls/sm ALL
boulder ALL
catalog ALL
cdos/business/nonprofit ALL
cdos/business/nonprofit Registration of Charities, Paid Solicitors, Professional Fundraising Consultants, and for-profit Public Benefit Corporations in Colorado
cdos/business/business Business Entities in Colorado
cdos/business/business Business Entity Transaction History
cdos/health ALL
cdos/lobbyist ALL
cdos/government ALL
cdos/business/business Master List in Colorado
cdos/business/business Trade Names for Businesses in Colorado
cdos/business/business Trademarks for Businesses in Colorado
cdos/business/ucc ALL
cdot/transportation_road_attributes ALL
cdot/transportation_infrastructure ALL
cdot/natural_resources ALL
cdot/tops ALL
dola/boundaries ALL
dola/special_districts ALL
dola/demographics ALL
cdor/revenue_marijuana ALL
cdor/retail_reports ALL
cdor/regulations_liquor ALL
ceo/useia Gasoline Prices in Colorado
ceo/useia Natural Gas Prices in Colorado
 AL

In [8]:
cronGroups["cdos/business/nonprofit"]["Registration of Charities, Paid Solicitors, Professional Fundraising Consultants, and for-profit Public Benefit Corporations in Colorado"]

'At 05:05 AM'

In [5]:
cronGroups["cdos/business/nonprofit"]

{'ALL': 'At 06:05 AM, only on Friday',
 'Registration of Charities, Paid Solicitors, Professional Fundraising Consultants, and for-profit Public Benefit Corporations in Colorado': 'At 05:05 AM'}

In [6]:
display(df[["Group","Title"]].head(200))

,Group,Title
0,catalog,CIM Catalog Download
1,cdos/business/business,Master List in Colorado
2,cdot/transportation_road_attributes,Highway Milepoints in Colorado
3,cdot/transportation_road_attributes,Highway Mileposts in Colorado
4,cdot/transportation_road_attributes,Highway Routes in Colorado
5,cdot/transportation_road_attributes,Highways in Colorado
6,cdot/transportation_road_attributes,Local Roads in Colorado
7,cdot/transportation_road_attributes,Major Roads in Colorado
8,cdot/transportation_road_attributes,Scenic Byways in Colorado
9,cdot/natural_resources,Lakes in Colorado


In [87]:
df.head()

,Title,Group,Program,Address,File,Count,Cron
0,CIM Catalog Download,catalog,general/scripts/request_url.js,https://data.colorado.gov/api/views/e7nm-tn2z/rows.csv?accessType=DOWNLOAD,catalog/data_source/inventory-$(date +%F).csv,1,At 04:00 AM
1,Master List in Colorado,cdos/business/business,general/scripts/request_url.js,http://coloradosos.gov/pubs/UCC/downloadFiles/UCCMstrLB1.txt,cdos/business/business/data_source/masterlist.tsv,1,"At 05:30 AM, on day 4 of the month"
2,Highway Milepoints in Colorado,cdot/transportation_road_attributes,general/scripts/request_url.js,http://dtdapps.coloradodot.info/staticdata/Downloads/StateGeoData/Mileposts.zip,cdot/transportation_road_attributes/data_source/mileposts.zip,1,"At 04:10 AM, on day 4 of the month"
3,Highway Mileposts in Colorado,cdot/transportation_road_attributes,general/scripts/request_url.js,http://dtdapps.coloradodot.info/staticdata/Downloads/StateGeoData/Mileposts.zip,cdot/transportation_road_attributes/data_source/mileposts.zip,1,"At 04:10 AM, on day 4 of the month"
4,Highway Routes in Colorado,cdot/transportation_road_attributes,general/scripts/request_url.js,http://dtdapps.coloradodot.info/staticdata/Downloads/StateGeoData/Routes.zip,cdot/transportation_road_attributes/data_source/routes.zip,1,"At 04:10 AM, on day 4 of the month"


In [114]:
newGroups={}
for group,titles in groups.items():
    spl=group.split("/")
    if len(spl) > 1:
       subGroup = '/'.join(spl[1:])
       group=spl[0]
    else:
       subGroup=""
    if group not in newGroups:
        newGroups[group]={}
    if len(subGroup) > 0:
        newGroups[group][subGroup]=titles
    else:
        newGroups[group]=titles


In [117]:
newGroups['cdot']

{'transportation_road_attributes': ['Highway Milepoints in Colorado',
  'Highway Mileposts in Colorado',
  'Highway Routes in Colorado',
  'Highways in Colorado',
  'Local Roads in Colorado',
  'Major Roads in Colorado',
  'Scenic Byways in Colorado'],
 'tops': ['CDOT Expenses', 'CDOT Revenues', 'CDOT Payroll Expenditures'],
 'natural_resources': ['Lakes in Colorado', 'Streams in Colorado'],
 'transportation_infrastructure': ['Airports in Colorado',
  'Cities in Colorado',
  'Counties in Colorado',
  'Railroads in Colorado']}

In [105]:
cronGroups

{'': {'ALL': 'At 02:47 PM', '': 'At 02:58 AM'},
 'boulder': {'ALL': 'At 03:10 AM'},
 'catalog': {'ALL': 'At 04:00 AM'},
 'cdos/business/nonprofit': {'ALL': 'At 06:05 AM, only on Friday',
  'Registration of Charities, Paid Solicitors, Professional Fundraising Consultants, and for-profit Public Benefit Corporations in Colorado': 'At 05:05 AM'},
 'cdos/business/business': {'Business Entities in Colorado': 'At 05:10 AM',
  'Business Entity Transaction History': 'At 08:00 AM',
  'Master List in Colorado': 'At 05:30 AM, on day 4 of the month',
  'Trade Names for Businesses in Colorado': 'At 04:00 AM, only on Tuesday',
  'Trademarks for Businesses in Colorado': 'At 04:00 AM, only on Tuesday'},
 'cdos/health': {'ALL': 'At 04:00 AM, only on Tuesday'},
 'cdos/lobbyist': {'ALL': 'At 04:15 AM'},
 'cdos/government': {'ALL': 'At 04:30 AM, only on Tuesday'},
 'cdos/business/ucc': {'ALL': 'At 03:00 AM'},
 'cdot/transportation_road_attributes': {'ALL': 'At 04:10 AM, on day 4 of the month'},
 'cdot/tran

In [95]:
tmp=df.loc[df["Title"] == "Master List in Colorado"]

In [96]:
tmp.shape

(1, 7)

In [103]:
df["Cron"].value_counts()

Cron
No Group Found                                       16
Unknown                                              14
At 04:15 AM                                          11
At 04:30 AM, on day 10 of the month                  10
At 05:20 AM, on day 4 of the month                   10
At 04:10 AM, on day 4 of the month                    7
At 04:40 AM, on day 6 of the month                    7
At 04:30 AM, on day 4 of the month                    4
At 04:00 AM, only on Tuesday                          4
At 03:00 AM                                           4
At 04:20 AM, on day 1, 8, 15, and 22 of the month     4
At 04:30 AM, on day 1 of the month                    3
At 08:45 AM, only on Tuesday                          3
At 04:00 AM, on day 4 of the month                    2
At 04:50 AM, on day 4 of the month                    2
At 03:10 AM                                           2
At 05:30 AM, on day 4 of the month                    1
At 04:00 AM                                

In [102]:
group=list(tmp["Group"].value_counts().to_dict().keys())[0]
prog=list(tmp["Program"].value_counts().to_dict().keys())[0]
cron=list(tmp["Cron"].value_counts().to_dict().keys())[0]
print(group,prog,cron)
add = tmp["Address"].values.tolist()
file = tmp["File"].values.tolist()


cdos/business/business general/scripts/request_url.js At 05:30 AM, on day 4 of the month


In [101]:
add

['http://coloradosos.gov/pubs/UCC/downloadFiles/UCCMstrLB1.txt']

In [20]:
df.loc[df["Address"].str.contains("ftps")]

,Title,Group,Program,Address,File,Count,Cron
37,Durable Medical Equipment Suppliers in Colorado,cdos/health,general/scripts/sftp_extract.js,ftps.sos.state.co.us,DME/CurrentDMESuppliers-CIM.txt,1,"At 04:00 AM, only on Tuesday"
38,Current Notaries in Colorado,cdos/government,general/scripts/sftp_extract.js,ftps.sos.state.co.us,notary/CurrentCommissionedNotaries.txt,1,"At 04:30 AM, only on Tuesday"
39,Uniform Commercial Code (UCC) Collateral Information in Colorado,cdos/business/ucc,general/scripts/sftp_extract.js,ftps.sos.state.co.us,UCC/ucccoll2.txt,1,At 03:00 AM
40,Uniform Commercial Code (UCC) Debtor Information in Colorado,cdos/business/ucc,general/scripts/sftp_extract.js,ftps.sos.state.co.us,UCC/uccdbtr.txt,1,At 03:00 AM
41,Uniform Commercial Code (UCC) Filing Information in Colorado,cdos/business/ucc,general/scripts/sftp_extract.js,ftps.sos.state.co.us,UCC/uccstmt.txt,1,At 03:00 AM
42,Secured Party Information in Colorado,cdos/business/ucc,general/scripts/sftp_extract.js,ftps.sos.state.co.us,UCC/uccsecr.txt,1,At 03:00 AM
43,Business Entities in Colorado,cdos/business/business,general/scripts/sftp_extract.js,ftps.sos.state.co.us,business/corpmstr.txt,1,At 05:10 AM
44,Business Entity Transaction History,cdos/business/business,general/scripts/sftp_extract.js,ftps.sos.state.co.us,business/corphist-2.txt,1,At 08:00 AM
45,Trademarks for Businesses in Colorado,cdos/business/business,general/scripts/sftp_extract.js,ftps.sos.state.co.us,business/trademarks.txt,1,"At 04:00 AM, only on Tuesday"
46,Trade Names for Businesses in Colorado,cdos/business/business,general/scripts/sftp_extract.js,ftps.sos.state.co.us,business/tradenames.txt,1,"At 04:00 AM, only on Tuesday"


In [ ]:
for group in df["Group"].value_counts().to_dict().keys():
    print(group)
    tmp = df.loc[df["Group"]==group]
    display(tmp[["Group","Title","Program","Address"]].head(100))